# Import dependencies

In [1]:
import numpy as np
import pandas as pd

print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")

NumPy: 2.5.4
pandas: 3.1.0


# Load dataset

In [7]:
# Since the dataset file is saved in the Strict Open XML Spreadsheet format, we
# have to use `calamine` to read it
df = pd.read_excel("dataset/dataset-uci.xlsx", engine="calamine")

# Shape of dataset; data types; feature overview; missing values; duplicates; etc.
rows = len(df)
print(f"Number of rows: {rows}")
#how many columns we have of data
columns = len(df.columns)
print(f"Number of columns: {columns}")
##Now the actual shape of the dataset + types
print(f"Dataset shape: {df.shape}")
print(f"Dataset datatypes:\n{df.dtypes}")

print(f"# null values for each feature:\n{df.isnull().sum()}")
print(f"# null values: {df.isnull().sum().sum()}")

print(f"# duplicated values: {df.duplicated().sum()}")

X = df["Gallstone Status"]
Y = df.drop(columns="Gallstone Status")

Number of rows: 319
Number of columns: 39
Dataset shape: (319, 39)
Dataset datatypes:
Gallstone Status                                    int64
Age                                                 int64
Gender                                              int64
Comorbidity                                         int64
Coronary Artery Disease (CAD)                       int64
Hypothyroidism                                      int64
Hyperlipidemia                                      int64
Diabetes Mellitus (DM)                              int64
Height                                              int64
Weight                                            float64
Body Mass Index (BMI)                             float64
Total Body Water (TBW)                            float64
Extracellular Water (ECW)                         float64
Intracellular Water (ICW)                         float64
Extracellular Fluid/Total Body Water (ECF/TBW)    float64
Total Body Fat Ratio (TBFR) (%)             

We have data of around 319 ppl.

In [8]:
#How many ppl have gallstones and how many do not
gallstone_counts = X.value_counts()[1]
print(f"Number of people with gallstones: {gallstone_counts}")
print(f"Number of people without gallstones: {len(X) - gallstone_counts}")


Number of people with gallstones: 158
Number of people without gallstones: 161


So we have half half ppl with and without gallstones

In [8]:
# Outlier
v_max = df.max()
v_mean = df.mean()
v_min = df.min()
v_std = df.std()

z = (df - v_mean) / v_std
outliers = z.abs() > 10

print(outliers.sum())               # number of outliers per column
print(df[outliers].stack().dropna())

Gallstone Status                                  0
Age                                               0
Gender                                            0
Comorbidity                                       0
Coronary Artery Disease (CAD)                     0
Hypothyroidism                                    0
Hyperlipidemia                                    0
Diabetes Mellitus (DM)                            0
Height                                            0
Weight                                            0
Body Mass Index (BMI)                             0
Total Body Water (TBW)                            0
Extracellular Water (ECW)                         0
Intracellular Water (ICW)                         0
Extracellular Fluid/Total Body Water (ECF/TBW)    0
Total Body Fat Ratio (TBFR) (%)                   0
Lean Mass (LM) (%)                                0
Body Protein Content (Protein) (%)                0
Visceral Fat Rating (VFR)                         0
Bone Mass (B

In [ ]:
# This is very high but clinically possible in poorly controlled diabetes
df.loc[40, "Diabetes Mellitus (DM)"]

np.int64(0)

In [ ]:
# This isn’t physiologically possible. It may be a misplaced decimal, such as 19.54 or 195.4. Obesity % is usually derived from BMI
df.loc[239, ["Height", "Weight", "Body Mass Index (BMI)", "Obesity (%)"]]

Height                    150.0
Weight                     65.9
Body Mass Index (BMI)      29.3
Obesity (%)              1954.0
Name: 239, dtype: float64

In [16]:
df.loc[318, ["Total Cholesterol (TC)", "Low Density Lipoprotein (LDL)", "High Density Lipoprotein (HDL)"]]

Total Cholesterol (TC)            296.0
Low Density Lipoprotein (LDL)     155.0
High Density Lipoprotein (HDL)    273.0
Name: 318, dtype: float64

In [17]:
# Highly correlated features
corr = Y.corr().abs()
upper = corr.where(np.triu(
    np.ones(corr.shape, dtype=bool),
    k=1
))  # upper triangle only

pairs = upper.stack()
high = pairs[pairs > 0.9]
print(high.sort_values(key=np.abs, ascending=False))

Total Body Fat Ratio (TBFR) (%)  Lean Mass (LM) (%)           0.994055
Total Body Water (TBW)           Muscle Mass (MM)             0.928633
                                 Extracellular Water (ECW)    0.904351
Intracellular Water (ICW)        Muscle Mass (MM)             0.904088
Body Mass Index (BMI)            Total Fat Content (TFC)      0.902706
dtype: float64


In [ ]:
# 